In [ ]:
from pathlib import Path 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import torch 
import json
from datetime import datetime

DATA_ROOT = Path("data/extracted")
DATA_ROOT.exists()

In [ ]:
symbol_dir = DATA_ROOT/"BAC_2026-04-01_2026-04-10_10"
orderbook_files = sorted(symbol_dir.glob("*orderbook_10.csv"))
message_files = sorted(symbol_dir.glob("*message_10.csv"))

In [ ]:
def read_message_file(file_path):
    message_cols = ["time","event_type","order_id","size","price","direction","extra"]
    df = pd.read_csv(file_path,names=message_cols)
    #drop extra column
    df = df.drop(columns=["extra"])
    return df
def read_orderbook_file(file_path):
    lab_cols = []
    for i in range(1,11):

        lab_cols.append(f"ask_price_{i}")
        lab_cols.append(f"ask_size_{i}")

        lab_cols.append(f"bid_price_{i}")
        lab_cols.append(f"bid_size_{i}")
    df = pd.read_csv(file_path,names=lab_cols)
    return df


def combine_message_orderbook(message_df,orderbook_df):

    #concat
    df = pd.concat([message_df,orderbook_df],axis=1)
    df_regular = df[
        (df['time'] > 10*60*60) &
        (df['time'] < 15.5*60*60) &
        (df['event_type'].isin([1,2,3,4,5]))
    ]
    return df_regular
message_df = read_message_file(message_files[0])
orderbook_df = read_orderbook_file(orderbook_files[0])
df = combine_message_orderbook(message_df,orderbook_df)

In [ ]:
#get midprice 
def get_smoothed_midprice_targets(df,k = 100):
    '''
    gets the smoothed midprice targets for the given dataframe
    and returns only the required columns (removing the message columns)
    '''
    df = df.iloc[:,6:]
    #get midprice
    df['midprice'] = (df['bid_price_1'] + df['ask_price_1'])/2
    #get smoothed midprice
    df['midprice_smooth'] = df['midprice'].rolling(window=k).mean()
    #get targets
    df['target'] = df['midprice_smooth'].shift(-k)/df['midprice_smooth'] - 1

    return df.dropna().drop(columns=['midprice','midprice_smooth'])
df = get_smoothed_midprice_targets(df)











In [ ]:
#get orderbook for each day and concat into tensors of 100 lookback
def get_lists_of_tensors(sorted_message_files:list[str],
                        sorted_orderbook_files:list[str],
                        ) -> tuple[list[torch.Tensor],list[torch.Tensor]]:
    #get orderbook for each day
    tensors_X = []
    tensors_y = []
    for message_file,orderbook_file in zip(sorted_message_files,sorted_orderbook_files):

        message_df = read_message_file(message_file)
        orderbook_df = read_orderbook_file(orderbook_file)
        df = combine_message_orderbook(message_df,orderbook_df)
        df = get_smoothed_midprice_targets(df)
        
        X = torch.tensor(df.iloc[:,:-1].values).float()
        y = torch.tensor(df.iloc[:,-1].values).float()
        X = X.unfold(0,100,10).permute(0,2,1)
        y = y.unfold(0,100,10)[:,-1]
        tensors_X.append(X)
        tensors_y.append(y)

        
    return tensors_X,tensors_y



    
Xs,ys = get_lists_of_tensors(message_files,orderbook_files)

X_train = torch.cat(Xs[:-2],dim=0)
y_train = torch.cat(ys[:-2],dim=0)
X_val = Xs[-2]
y_val = ys[-2]
X_test = Xs[-1]
y_test = ys[-1]






In [ ]:
def normalize_train_val_test(X_train,X_val,X_test):
    #normalize X_train per feature
    X_train_mean = X_train.mean(dim=0)
    X_train_std = X_train.std(dim=0)
    X_train = (X_train - X_train_mean) / X_train_std
    X_val = (X_val - X_train_mean) / X_train_std
    X_test = (X_test - X_train_mean) / X_train_std
    return X_train,X_val,X_test

def make_target_labels(y_train,y_val,y_test):
    #make into balanced ternary labels

    #get thresholds
    low_threshold = torch.quantile(y_train,0.33)
    high_threshold = torch.quantile(y_train,0.66)

    y_train = torch.where(y_train < low_threshold,0,
                          torch.where(y_train > high_threshold,2,1))
    y_val = torch.where(y_val < low_threshold,0,
                        torch.where(y_val > high_threshold,2,1))
    y_test = torch.where(y_test < low_threshold,0,
                         torch.where(y_test > high_threshold,2,1))
    return y_train.long(),y_val.long(),y_test.long()

X_train,X_val,X_test = normalize_train_val_test(X_train,X_val,X_test)
y_train,y_val,y_test = make_target_labels(y_train,y_val,y_test)

In [ ]:
from torch.utils.data import TensorDataset,DataLoader
import torch.nn as nn
import torch.nn.functional as F

batch_size = 1024

X_train = X_train.unsqueeze(1)
X_val = X_val.unsqueeze(1)
X_test = X_test.unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train,y_train),batch_size=batch_size,shuffle=True)
val_loader = DataLoader(TensorDataset(X_val,y_val),batch_size=batch_size,shuffle=False)
test_loader = DataLoader(TensorDataset(X_test,y_test),batch_size=batch_size,shuffle=False)

class DeepLOB(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(1, 2), stride=(1, 2)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=(1, 2), stride=(1, 2)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=(1, 10)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(32),
        )

        self.inp1 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=(1, 1), padding="same"),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=(3, 1), padding="same"),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(64),
        )

        self.inp2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=(1, 1), padding="same"),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=(5, 1), padding="same"),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(64),
        )

        self.inp3 = nn.Sequential(
            nn.MaxPool2d(kernel_size=(3, 1), stride=(1, 1), padding=(1, 0)),
            nn.Conv2d(32, 64, kernel_size=(1, 1), padding="same"),
            nn.LeakyReLU(0.01),
            nn.BatchNorm2d(64),
        )

        self.lstm = nn.LSTM(input_size=192, hidden_size=64, batch_first=True)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)

        x = torch.cat([self.inp1(x), self.inp2(x), self.inp3(x)], dim=1)

        # (N, 192, T', 1) -> (N, T', 192)
        x = x.permute(0, 2, 1, 3).squeeze(-1)

        x, _ = self.lstm(x)
        x = x[:, -1, :]
        return self.fc(x)

def MLPLOB(nn.module):
    